# Internet of Things Application Development
# Lab 4 - AI in IoT application
Task: Your task is to setup and build an ML/DL model to process and predict temperature and humidity data taken from your sensors.

## Convolution Neural Network singular variable model
This is an example to predict the humidity given a sequence of humidity data

In [14]:
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauth
from google.colab import drive
import numpy as np
import pandas as pd
import random
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Flatten
from keras.layers import Conv1D
from keras.layers import MaxPooling1D
from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io

  Using cached google_api_python_client-2.166.0-py2.py3-none-any.whl.metadata (6.6 kB)
ERROR: Could not find a version that satisfies the requirement google-auth-oauth (from versions: none)
ERROR: No matching distribution found for google-auth-oauth


In [19]:
# Data preprocessing
def split_sequence(sequence, n_steps):
	X, y = list(), list()
	for i in range(len(sequence)):
		# Find the end of this pattern
		end_ix = i + n_steps
		# Check if we are beyond the sequence
		if end_ix > len(sequence)-1:
			break
		# Gather input and output parts of the pattern
		seq_x, seq_y = sequence[i:end_ix], sequence[end_ix]
		X.append(seq_x)
		y.append(seq_y)
	return np.array(X), np.array(y)

In [20]:
# Function to read csv file from google drive
def read_csv_from_drive_link(file_id):
  drive_service = build('drive', 'v3')
  request = drive_service.files().get_media(fileId=file_id)
  downloaded = io.BytesIO()
  downloader = MediaIoBaseDownload(downloaded, request)
  done = False
  while done is False:
    status, done = downloader.next_chunk()

  downloaded.seek(0)
  return pd.read_csv(downloaded)

In [28]:
# Read given train and test sets
train_data = read_csv_from_drive_link("1CvX-gjGkiPDojF9vG9VQVJ5sBxNCOG1P")
test_data = read_csv_from_drive_link("1Kvv5EBQFad5PfoEkSgiMKVT1dE-9BglZ")
humi_seq_train = train_data['Relative_humidity_room'].tolist()
humi_seq_test = test_data['Relative_humidity_room'].tolist()

In [29]:
# Preprocessing steps
# Choose a number of time steps
n_steps = 3
# Split into samples
X, y = split_sequence(humi_seq_train, n_steps)
# Reshape from [samples, timesteps] into [samples, timesteps, features]
n_features = 1
X = X.reshape((X.shape[0], X.shape[1], n_features))

In [30]:
# Define model
model = Sequential()
model.add(Conv1D(filters=64, kernel_size=2, activation='relu', input_shape=(n_steps, n_features)))
model.add(MaxPooling1D(pool_size=2))
model.add(Flatten())
model.add(Dense(50, activation='relu'))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [31]:
# Fit model
model.fit(X, y, epochs=200, verbose=1)

Epoch 1/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 470.1795
Epoch 2/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3531
Epoch 3/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3047
Epoch 4/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.2631
Epoch 5/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2602
Epoch 6/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.2865
Epoch 7/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3005
Epoch 8/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2777
Epoch 9/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3124
Epoch 10/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.2883
Epoch 11/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2980
Epoch 12/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2900
Epoch 13/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2387
Epoch 14/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.3089
Epoch 15/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.2866
Ep

In [32]:
# Predict on test set based on the steps
for i in range(10):
  random_num = random.randint(0, len(humi_seq_test)-4)
  x_input = np.array(humi_seq_test[random_num:random_num+n_steps])
  x_input = x_input.reshape((1, n_steps, n_features))
  predicted_value = model.predict(x_input, verbose=0)
  rmse = np.sqrt(np.mean((humi_seq_test[random_num+n_steps] - predicted_value[0][0])**2))
  print("Sequence:", np.array(humi_seq_test[random_num:random_num+n_steps]),"Next value:", humi_seq_test[random_num+n_steps], ", Predicted next value:", predicted_value[0][0], ", RMSE:", rmse)

Sequence: [31.5667 31.468  31.2933] Next value: 31.1627 , Predicted next value: 31.256233 , RMSE: 0.09353256
Sequence: [28.24   28.1413 28.1453] Next value: 28.0907 , Predicted next value: 28.0394 , RMSE: 0.05130005
Sequence: [33.4867 33.584  33.656 ] Next value: 33.656 , Predicted next value: 33.626484 , RMSE: 0.029514313
Sequence: [41.756  41.5667 41.2227] Next value: 41.036 , Predicted next value: 41.20652 , RMSE: 0.17052078
Sequence: [27.652  27.6453 27.508 ] Next value: 27.3467 , Predicted next value: 27.493279 , RMSE: 0.14657784
Sequence: [41.728  41.2213 40.9133] Next value: 40.552 , Predicted next value: 40.717934 , RMSE: 0.16593552
Sequence: [28.588  28.4307 28.324 ] Next value: 28.4347 , Predicted next value: 28.226217 , RMSE: 0.20848274
Sequence: [38.4813 38.5947 38.6387] Next value: 38.716 , Predicted next value: 38.5977 , RMSE: 0.11830139
Sequence: [41.04  41.936 41.756] Next value: 41.5667 , Predicted next value: 42.162148 , RMSE: 0.59544754
Sequence: [42.096  41.8507 41.

## The multivariate model
This is an example to predict two different values (humidity & CO2) in 1 multivariate model

In [26]:
# Split a multivariate sequence into samples
def split_sequences(sequences, n_steps):
	X, y = list(), list()
	for i in range(len(sequences)):
		# Find the end of this pattern
		end_ix = i + n_steps
		# Check if we are beyond the dataset
		if end_ix > len(sequences)-1:
			break
		# Gather input and output parts of the pattern
		seq_x, seq_y = sequences[i:end_ix, :], sequences[end_ix, :]
		X.append(seq_x)
		y.append(seq_y)
	return np.array(X), np.array(y)

In [39]:
# Read given CO2 data in train and test sets
humi_seq_train = np.array(train_data['Relative_humidity_room'])
humi_seq_test = np.array(test_data['Relative_humidity_room'])
co2_seq_train = np.array(train_data['CO2_room'])
co2_seq_test = np.array(test_data['CO2_room'])

In [41]:
# Preprocessing steps
# Convert to [rows, columns] structure
humi_seq_train = humi_seq_train.reshape((len(humi_seq_train), 1))
co2_seq_train = co2_seq_train.reshape((len(co2_seq_train), 1))
# Horizontally stack columns
dataset = np.hstack((humi_seq_train, co2_seq_train))
# Choose a number of time steps
n_steps = 3
# Convert into input/output
X, y = split_sequences(dataset, n_steps)
# The dataset knows the number of features
n_features = X.shape[2]

In [42]:
# Define model
model = Sequential()
model.add(Conv1D(filters=64, kernel_size=2, activation='relu', input_shape=(n_steps, n_features)))
model.add(MaxPooling1D(pool_size=2))
model.add(Flatten())
model.add(Dense(50, activation='relu'))
model.add(Dense(n_features))
model.compile(optimizer='adam', loss='mse')

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [43]:
# Fit model
model.fit(X, y, epochs=200, verbose=1)

Epoch 1/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 6377.4399
Epoch 2/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 124.8899
Epoch 3/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 92.1442
Epoch 4/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 93.7452
Epoch 5/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 53.1363
Epoch 6/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 77.3394
Epoch 7/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 72.9840
Epoch 8/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 68.5874
Epoch 9/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 55.3441
Epoch 10/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 60.6722
Epoch 11/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 47.7705
Epoch 12/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 46.8123
Epoch 13/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 45.8186
Epoch 14/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 32.7799
Epoch 15/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - 

In [66]:
# Predict on test set based on the steps
for i in range(10):
  random_num = random.randint(0, len(test_data)-4)
  x_input = np.vstack((humi_seq_test[random_num:random_num+n_steps], co2_seq_test[random_num:random_num+n_steps])).T
  x_input = x_input.reshape((1, n_steps, n_features))
  predicted_value = model.predict(x_input, verbose=0)
  rmse_humi = np.sqrt(np.mean((humi_seq_test[random_num+n_steps] - predicted_value[0][0])**2))
  rmse_co2 = np.sqrt(np.mean((co2_seq_test[random_num+n_steps] - predicted_value[0][1])**2))
  print("Sequence", i)
  print("Humidity sequence:", np.array(humi_seq_test[random_num:random_num+n_steps]), ", Next value:", humi_seq_test[random_num+n_steps], ", Predicted humidity:", predicted_value[0][0], ", RMSE:", rmse_humi)
  print("CO2 sequence:", np.array(co2_seq_test[random_num:random_num+n_steps]), ", Next value:", co2_seq_test[random_num+n_steps], ", Predicted CO2:", predicted_value[0][1], ", RMSE:", rmse_co2)
  print("-"*20)

Sequence 0
Humidity sequence: [31.3547 31.416  31.4933] , Next value: 31.544 , Predicted humidity: 31.455647 , RMSE: 0.08835348510742236
CO2 sequence: [206.123 206.229 207.093] , Next value: 206.059 , Predicted CO2: 207.61977 , RMSE: 1.560766235351565
--------------------
Sequence 1
Humidity sequence: [42.7627 42.9387 43.0987] , Next value: 43.5573 , Predicted humidity: 42.78107 , RMSE: 0.7762292907714823
CO2 sequence: [215.701 218.155 217.003] , Next value: 216.459 , Predicted CO2: 219.00699 , RMSE: 2.547988525390622
--------------------
Sequence 2
Humidity sequence: [27.6213 27.652  27.6453] , Next value: 27.508 , Predicted humidity: 27.665894 , RMSE: 0.15789355468750088
CO2 sequence: [205.696 205.109 204.469] , Next value: 204.139 , Predicted CO2: 203.73169 , RMSE: 0.40731054687501
--------------------
Sequence 3
Humidity sequence: [41.9893 42.16   42.184 ] , Next value: 42.1653 , Predicted humidity: 41.91353 , RMSE: 0.2517715576171895
CO2 sequence: [197.643 197.963 197.899] , Next 